# Stage 07-C: Supervised Fine-Tuning with Frozen PhoBERT

**Purpose:** Train COOLANT's cross-modal fusion (§3.3) and attention-guided aggregation (§3.4)
on top of a **fully frozen** `vinai/phobert-base-v2` text encoder on labelled ViFactCheck-MM.
PhoBERT is used as a fixed feature extractor — zero gradient flows through it.

## Why frozen?

- Avoids catastrophic forgetting of Vietnamese language knowledge
- Only the projection heads, fusion, and classifier are trained (~6M params vs 130M)  
- Faster training, more stable, no need for `07b` contrastive pre-training

## Architecture

```
Statement + Context
     |
PhoBERT-base-v2  ← FROZEN (all 12 layers, no gradient)
  CLS [B, 768] → TextProjector [768→128] → e_t [B, 128]
     |
     +── CrossModalFusion §3.3 (multi-head attention) ──+
     |                                                   |
  image [B, max_pairs, 64] → ImageProjector [64→128] → e_v [B, P, 128]
     |
  AttentionGuidedAggregation §3.4 (VAE + SE-Net)
     |
  Classifier MLP [256→128→3]  (Real=0, Fake=1, NEI=2)

Trainable only: TextProjector, ImageProjector, ITMHead, CMF, VAE, SENet, Classifier
```

## Loss (COOLANT Eq. 19)

```
L = L_CLS + L_ITM + L_CL + γ·L_CA
```

## Prerequisites

- `06d_prepare_dataset.ipynb` built `stage06d_cache/{train,dev,test}.h5`
- `07b` is **NOT required** (PhoBERT is not fine-tuned)

## Outputs

- `training/checkpoints_stage07c/best_finetune.pt`
- `training/stage07c_results/finetune_results.json`
- `training/stage07c_results/training_curves.png`

In [ ]:
# ─── Environment Setup (do not edit) ─────────────────────────────────────────
import os, sys
from pathlib import Path


def _detect_platform():
    try:
        import google.colab  # noqa: F401
        return 'colab', False
    except ImportError:
        pass
    if Path('/workspace').exists() and os.environ.get('VAST_CONTAINERLABEL'):
        return 'vastai', False
    if Path('/workspace').exists():
        return 'vastai', True
    if sys.platform == 'win32': return 'windows', False
    if sys.platform == 'darwin': return 'mac', False
    return None, True


PLATFORM, _uncertain = _detect_platform()
if PLATFORM == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')

try:
    _nb_path = Path(__file__).resolve()
except NameError:
    _nb_path = Path.cwd()

PROJECT_ROOT = (
    Path('/content/drive/MyDrive/Thesis_Final/fake-news-detection') if PLATFORM == 'colab'
    else _nb_path.parents[1]
)
sys.path.insert(0, str(PROJECT_ROOT))

_env_map = {
    'colab':   PROJECT_ROOT / '.env.colab',
    'vastai':  PROJECT_ROOT / '.env.vastai',
    'windows': PROJECT_ROOT / '.env.windows',
    'mac':     PROJECT_ROOT / '.env.mac',
}
_env_file = _env_map.get(PLATFORM, PROJECT_ROOT / '.env')
if not _env_file.exists():
    _env_file = PROJECT_ROOT / '.env'

from dotenv import load_dotenv
load_dotenv(_env_file, override=True)
from src.utils.env_utils import get_data_root
DATA_ROOT = get_data_root()
print(f'Platform : {PLATFORM}  DATA_ROOT: {DATA_ROOT}')

In [ ]:
# ── Step 1: Configuration ─────────────────────────────────────────────────────
CONFIG = {
    'paths': {
        'stage06d_cache_dir': DATA_ROOT / 'training' / 'stage06d_cache',
        'checkpoint_dir':     DATA_ROOT / 'training' / 'checkpoints_stage07c',
        'results_dir':        DATA_ROOT / 'training' / 'stage07c_results',
        'mlflow_dir':         DATA_ROOT / 'mlruns',
    },
    'phobert': {
        'model_name':   'vinai/phobert-base-v2',
        'max_seq_len':  256,
        'text_a_field': 'statement',
        'text_b_field': 'context',
    },
    'model': {
        'image_dim':   64,
        'text_dim':    768,
        'proj_dim':    128,
        'temperature': 0.07,
        'itm_hidden':  256,
        'attn_heads':  4,
        'h_dim':       64,
        'num_classes': 3,
        'dropout':     0.1,
        'max_pairs':   32,
    },
    'loss': {
        'lambda_itm':      1.0,
        'lambda_cl':       1.0,
        'gamma_ca':        0.5,
        'label_smoothing': 0.05,
    },
    'training': {
        'batch_size':   32,
        'max_epochs':   30,
        'patience':     5,
        'head_lr':      1e-4,   # only heads are trained — single LR group
        'weight_decay': 0.01,
        'warmup_pct':   0.06,
        'grad_clip':    1.0,
        'seed':         42,
    },
    'safety': {'smoke_test': False, 'smoke_batches': 3, 'smoke_epochs': 2},
}
for d in ('checkpoint_dir', 'results_dir'):
    CONFIG['paths'][d].mkdir(parents=True, exist_ok=True)

print('PhoBERT : FROZEN (feature extractor only)')
print(f'head_lr={CONFIG["training"]["head_lr"]}  gamma_ca={CONFIG["loss"]["gamma_ca"]}')

In [ ]:
# ── Step 2: Dependencies, Device, Seed ────────────────────────────────────────
import importlib, unicodedata, random, json, gc
from datetime import datetime

_required = [
    ('torch', 'torch'), ('h5py', 'h5py'), ('numpy', 'numpy'),
    ('pandas', 'pandas'), ('matplotlib', 'matplotlib'),
    ('seaborn', 'seaborn'), ('tqdm', 'tqdm'),
    ('transformers', 'transformers'), ('sklearn', 'scikit-learn'),
]
_missing = [pkg for mod, pkg in _required if importlib.util.find_spec(mod) is None]
if _missing:
    raise RuntimeError(f'Missing packages: {_missing}')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import h5py
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, classification_report
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup


def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)


seed_everything(CONFIG['training']['seed'])
device = torch.device('cuda' if torch.cuda.is_available() else
                      'mps'  if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {device}  PyTorch: {torch.__version__}')

In [ ]:
# ── Step 3: Load Labelled Dataset ─────────────────────────────────────────────
LABEL_NAMES = {0: 'Real', 1: 'Fake', 2: 'NEI'}


def load_split(split, cfg):
    p = cfg['paths']['stage06d_cache_dir'] / f'{split}.h5'
    if not p.exists():
        raise FileNotFoundError(f'{p}\nRun 06d_prepare_dataset.ipynb first.')
    recs = []
    max_p = cfg['model']['max_pairs']
    with h5py.File(p, 'r') as f:
        n         = int(f.attrs['n_articles'])
        cache_mp  = int(f.attrs['max_pairs'])
        if cache_mp != max_p:
            raise ValueError(f'max_pairs mismatch: cache={cache_mp} CONFIG={max_p}')
        imgs  = f['image_features'][:]   # [N, max_pairs, 64]
        cnts  = f['pair_counts'][:]       # [N]
        lbls  = f['labels'][:]            # [N]
        stmts = [str(s) for s in f['statements'][:]]
        ctxs  = [str(s) for s in f['contexts'][:]]
    for i in range(n):
        recs.append({
            'statement':       unicodedata.normalize('NFC', stmts[i]),
            'context':         unicodedata.normalize('NFC', ctxs[i]),
            'image':           imgs[i].astype(np.float32),   # [max_pairs, 64]
            'pair_count':      int(cnts[i]),
            'label':           int(lbls[i]),
            'has_valid_image': int(cnts[i]) > 0,
        })
    return recs


data = {s: load_split(s, CONFIG) for s in ('train', 'dev', 'test')}
for split, recs in data.items():
    hist = np.bincount([r['label'] for r in recs], minlength=3).tolist()
    vi   = sum(1 for r in recs if r['has_valid_image'])
    print(f'  [{split}] n={len(recs)}  R/F/N={hist}  with_image={vi} ({100*vi//len(recs)}%)')

In [ ]:
# ── Step 4: PhoBERT Tokenizer ─────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(
    CONFIG['phobert']['model_name'], use_fast=True
)
print(f'Tokenizer: {CONFIG["phobert"]["model_name"]}  vocab={tokenizer.vocab_size}')

In [ ]:
# ── Step 5: Dataset + DataLoader ──────────────────────────────────────────────
class ViFactCheckMMDataset(Dataset):
    """Labelled ViFactCheck-MM with text-only fallback for no-image instances."""
    def __init__(self, records, tokenizer, max_len, text_a='statement', text_b='context'):
        self.records   = records
        self.tokenizer = tokenizer
        self.max_len   = max_len
        self.text_a    = text_a
        self.text_b    = text_b

    def __len__(self): return len(self.records)

    def __getitem__(self, idx):
        r = self.records[idx]
        enc = self.tokenizer(
            r[self.text_a], r[self.text_b],
            max_length=self.max_len,
            padding='max_length',
            truncation='only_second',
            return_tensors='pt',
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'image':          torch.tensor(r['image'], dtype=torch.float32),  # [max_pairs, 64]
            'pair_count':     torch.tensor(r['pair_count'], dtype=torch.long),
            'has_image':      torch.tensor(float(r['has_valid_image'])),
            'label':          torch.tensor(r['label'], dtype=torch.long),
        }


max_len = CONFIG['phobert']['max_seq_len']
bs      = CONFIG['training']['batch_size']

datasets = {
    s: ViFactCheckMMDataset(data[s], tokenizer, max_len,
                            CONFIG['phobert']['text_a_field'],
                            CONFIG['phobert']['text_b_field'])
    for s in ('train', 'dev', 'test')
}
loaders = {
    'train': DataLoader(datasets['train'], batch_size=bs, shuffle=True,  num_workers=2, pin_memory=True),
    'dev':   DataLoader(datasets['dev'],   batch_size=bs, shuffle=False, num_workers=2, pin_memory=True),
    'test':  DataLoader(datasets['test'],  batch_size=bs, shuffle=False, num_workers=2, pin_memory=True),
}
print(f'Train={len(loaders["train"])} batches  Val={len(loaders["dev"])}  Test={len(loaders["test"])}')

In [ ]:
# ── Step 6: Full COOLANT Model (Stage 2) ─────────────────────────────────────
class CrossModalFusion(nn.Module):
    """COOLANT §3.3: inter-modal attention + outer-product interaction."""
    def __init__(self, dim, heads, dropout):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(dim)

    def forward(self, q, kv, key_padding_mask=None):
        # q: [B, 1, dim]  kv: [B, P, dim]
        attended, _ = self.attn(q, kv, kv, key_padding_mask=key_padding_mask)
        fused = self.norm(q + attended)   # residual
        return fused.squeeze(1)           # [B, dim]


class AmbiguityVAE(nn.Module):
    """COOLANT §3.4: VAE-based ambiguity scoring (Eq. 11-14)."""
    def __init__(self, dim, h_dim):
        super().__init__()
        self.enc_mu  = nn.Linear(dim, h_dim)
        self.enc_log = nn.Linear(dim, h_dim)
        self.dec     = nn.Linear(h_dim, dim)

    def forward(self, x):
        mu      = self.enc_mu(x)
        log_var = self.enc_log(x)
        std     = torch.exp(0.5 * log_var)
        z       = mu + std * torch.randn_like(std) if self.training else mu
        recon   = self.dec(z)
        kl_div  = -0.5 * (1 + log_var - mu.pow(2) - log_var.exp()).mean()
        return recon, kl_div


class ViCOOLANTFull(nn.Module):
    """
    Full COOLANT re-instantiated with PhoBERT text tower.
    Loss: L = L_ITM + λ·L_CL + γ·L_CA  (COOLANT paper Eq. 19)
    Text-only branch: instances with has_image=False skip L_ITM and L_CL.
    """
    def __init__(self, phobert_name, image_dim, text_dim, proj_dim, temperature,
                 itm_hidden, attn_heads, h_dim, num_classes, dropout, max_pairs):
        super().__init__()
        self.temperature = temperature
        self.max_pairs   = max_pairs

        # §3.2 — encoders + projections (loaded from Stage 1 checkpoint)
        self.text_encoder = AutoModel.from_pretrained(phobert_name)
        self.text_proj = nn.Sequential(
            nn.Linear(text_dim, proj_dim), nn.LayerNorm(proj_dim), nn.GELU()
        )
        self.image_proj = nn.Sequential(
            nn.Linear(image_dim, proj_dim), nn.LayerNorm(proj_dim), nn.GELU()
        )
        self.itm_head = nn.Sequential(
            nn.Linear(proj_dim * 3, itm_hidden), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(itm_hidden, 1)
        )

        # §3.3 — cross-modal fusion
        self.cmf = CrossModalFusion(proj_dim, attn_heads, dropout)

        # §3.4 — attention-guided aggregation
        self.vae    = AmbiguityVAE(proj_dim, h_dim)
        self.se_fc1 = nn.Linear(proj_dim, proj_dim // 2)
        self.se_fc2 = nn.Linear(proj_dim // 2, proj_dim)

        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(proj_dim * 2, 256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, 128),          nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, num_classes),
        )

    def _se_attention(self, x):
        """SE-Net-style channel attention (Eq. 14-ish)."""
        s = F.adaptive_avg_pool1d(x.unsqueeze(1), 1).squeeze(1)
        s = torch.sigmoid(self.se_fc2(F.relu(self.se_fc1(s))))
        return x * s

    def encode_text(self, input_ids, attention_mask):
        out = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        return F.normalize(self.text_proj(cls), dim=-1)

    def encode_image(self, image, pair_counts):
        """Project each pair; return [B, P, proj_dim] and padding mask."""
        B, P, _ = image.shape
        e_v_all  = F.normalize(self.image_proj(image.view(B * P, -1)), dim=-1)
        e_v_all  = e_v_all.view(B, P, -1)  # [B, P, proj_dim]
        # Build key_padding_mask: True = padded position
        mask = torch.ones(B, P, dtype=torch.bool, device=image.device)
        for i, pc in enumerate(pair_counts.tolist()):
            if pc > 0:
                mask[i, :pc] = False
        return e_v_all, mask

    def itc_loss(self, e_t, e_v_mean):
        logits = (e_v_mean @ e_t.T) / self.temperature
        labels = torch.arange(len(e_t), device=e_t.device)
        return (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels)) / 2

    def itm_loss(self, e_t, e_v_mean):
        B = e_t.size(0)
        pos = torch.cat([e_t, e_v_mean, (e_t - e_v_mean).abs()], dim=-1)
        pos_l = self.itm_head(pos).squeeze(-1)
        neg_idx = torch.randperm(B, device=e_t.device)
        neg = torch.cat([e_t, e_v_mean[neg_idx], (e_t - e_v_mean[neg_idx]).abs()], dim=-1)
        neg_l = self.itm_head(neg).squeeze(-1)
        return F.binary_cross_entropy_with_logits(
            torch.cat([pos_l, neg_l]),
            torch.cat([torch.ones(B, device=e_t.device), torch.zeros(B, device=e_t.device)]),
        )

    def forward(self, input_ids, attention_mask, image, pair_counts, has_image):
        B = input_ids.size(0)
        e_t      = self.encode_text(input_ids, attention_mask)  # [B, proj_dim]
        e_v_all, pad_mask = self.encode_image(image, pair_counts)  # [B, P, proj_dim]

        img_mask = has_image.bool()  # [B]

        # Contrastive losses — only for image-present instances
        l_itc = l_itm = torch.tensor(0.0, device=e_t.device)
        if img_mask.sum() >= 2:
            e_t_img     = e_t[img_mask]
            e_v_mean    = e_v_all[img_mask].mean(dim=1)  # [M, proj_dim]
            e_v_mean    = F.normalize(e_v_mean, dim=-1)
            l_itc = self.itc_loss(e_t_img, e_v_mean)
            l_itm = self.itm_loss(e_t_img, e_v_mean)

        # Cross-modal fusion — §3.3
        q        = e_t.unsqueeze(1)                      # [B, 1, proj_dim]
        fused    = self.cmf(q, e_v_all, key_padding_mask=pad_mask)  # [B, proj_dim]

        # Zero fusion output for text-only instances (attention-guidance zeroed)
        fused = fused * has_image.unsqueeze(-1)

        # Attention-guided aggregation — §3.4
        recon, kl_div = self.vae(fused)
        fused_se      = self._se_attention(fused)
        # L_CA: reconstruction + KL (attention guidance)
        l_ca = F.mse_loss(recon, fused.detach()) + kl_div

        # Final representation: concat text + fused visual
        h = torch.cat([e_t, fused_se], dim=-1)  # [B, proj_dim*2]
        logits = self.classifier(h)              # [B, num_classes]

        return logits, l_itc, l_itm, l_ca


model = ViCOOLANTFull(
    phobert_name=CONFIG['phobert']['model_name'],
    image_dim=CONFIG['model']['image_dim'],
    text_dim=CONFIG['model']['text_dim'],
    proj_dim=CONFIG['model']['proj_dim'],
    temperature=CONFIG['model']['temperature'],
    itm_hidden=CONFIG['model']['itm_hidden'],
    attn_heads=CONFIG['model']['attn_heads'],
    h_dim=CONFIG['model']['h_dim'],
    num_classes=CONFIG['model']['num_classes'],
    dropout=CONFIG['model']['dropout'],
    max_pairs=CONFIG['model']['max_pairs'],
).to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Params: trainable={trainable:,}  total={total:,}')

In [ ]:
# ── Step 7: Freeze PhoBERT ────────────────────────────────────────────────────
for p in model.text_encoder.parameters():
    p.requires_grad = False

frozen   = sum(p.numel() for p in model.text_encoder.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'PhoBERT frozen : {frozen:,} params')
print(f'Trainable      : {trainable:,} params  (heads + fusion + classifier)')
print(f'Total          : {total:,} params')

In [ ]:
# ── Step 8: Optimizer + Scheduler ────────────────────────────────────────────
# PhoBERT is frozen — single optimizer group covering projection heads,
# fusion, VAE, SE-Net, ITM head, and classifier only.
head_params = [p for p in model.parameters() if p.requires_grad]

optimizer = torch.optim.AdamW(
    head_params,
    lr=CONFIG['training']['head_lr'],
    weight_decay=CONFIG['training']['weight_decay'],
)

total_steps  = len(loaders['train']) * CONFIG['training']['max_epochs']
warmup_steps = int(total_steps * CONFIG['training']['warmup_pct'])
scheduler    = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)

cls_criterion = nn.CrossEntropyLoss(label_smoothing=CONFIG['loss']['label_smoothing'])
print(f'Trainable params in optimizer: {sum(p.numel() for p in head_params):,}')
print(f'Total steps={total_steps}  warmup={warmup_steps}')

In [ ]:
# ── Step 9: Evaluation Function ───────────────────────────────────────────────
@torch.no_grad()
def evaluate(model, loader, loss_cfg):
    model.eval()
    all_preds, all_labels = [], []
    total_loss, n_batches = 0.0, 0

    for batch in loader:
        ids   = batch['input_ids'].to(device)
        mask  = batch['attention_mask'].to(device)
        imgs  = batch['image'].to(device)
        pcnts = batch['pair_count'].to(device)
        hi    = batch['has_image'].to(device)
        lbls  = batch['label'].to(device)

        logits, l_itc, l_itm, l_ca = model(ids, mask, imgs, pcnts, hi)
        l_cls = cls_criterion(logits, lbls)
        loss  = (l_cls
                 + loss_cfg['lambda_itm'] * l_itm
                 + loss_cfg['lambda_cl']  * l_itc
                 + loss_cfg['gamma_ca']   * l_ca)
        total_loss += loss.item()
        n_batches  += 1

        preds = logits.argmax(dim=-1).cpu().numpy()
        all_preds.extend(preds.tolist())
        all_labels.extend(lbls.cpu().numpy().tolist())

    macro_f1  = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    accuracy  = accuracy_score(all_labels, all_preds)
    f1_per_cls = f1_score(all_labels, all_preds, average=None, zero_division=0, labels=[0, 1, 2])
    cm = confusion_matrix(all_labels, all_preds, labels=[0, 1, 2]).tolist()

    return {
        'loss':       total_loss / max(n_batches, 1),
        'macro_f1':   macro_f1,
        'accuracy':   accuracy,
        'f1_real':    float(f1_per_cls[0]),
        'f1_fake':    float(f1_per_cls[1]),
        'f1_nei':     float(f1_per_cls[2]),
        'confusion_matrix': cm,
        'preds':      all_preds,
        'labels':     all_labels,
    }


print('evaluate() defined')

In [ ]:
# ── Step 10: Training Loop ────────────────────────────────────────────────────
loss_cfg    = CONFIG['loss']
max_epochs  = CONFIG['training']['max_epochs']
patience    = CONFIG['training']['patience']
grad_clip   = CONFIG['training']['grad_clip']
smoke       = CONFIG['safety']['smoke_test']
smoke_b     = CONFIG['safety']['smoke_batches']
smoke_ep    = CONFIG['safety']['smoke_epochs']
ckpt_path   = CONFIG['paths']['checkpoint_dir'] / 'best_finetune.pt'

history      = []
best_f1      = 0.0
no_improve   = 0
_max_epochs  = smoke_ep if smoke else max_epochs

for epoch in range(1, _max_epochs + 1):
    model.train()
    train_loss, train_steps = 0.0, 0

    pbar = tqdm(loaders['train'], desc=f'Ep {epoch}/{_max_epochs}', leave=False)
    for step, batch in enumerate(pbar):
        if smoke and step >= smoke_b: break
        ids   = batch['input_ids'].to(device)
        mask  = batch['attention_mask'].to(device)
        imgs  = batch['image'].to(device)
        pcnts = batch['pair_count'].to(device)
        hi    = batch['has_image'].to(device)
        lbls  = batch['label'].to(device)

        optimizer.zero_grad()
        logits, l_itc, l_itm, l_ca = model(ids, mask, imgs, pcnts, hi)
        l_cls = cls_criterion(logits, lbls)
        loss  = (l_cls
                 + loss_cfg['lambda_itm'] * l_itm
                 + loss_cfg['lambda_cl']  * l_itc
                 + loss_cfg['gamma_ca']   * l_ca)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()
        scheduler.step()

        train_loss  += loss.item()
        train_steps += 1
        pbar.set_postfix(loss=f'{loss.item():.3f}')

    val_m = evaluate(model, loaders['dev'], loss_cfg)
    avg_train = train_loss / max(train_steps, 1)

    row = {
        'epoch':        epoch,
        'train_loss':   avg_train,
        'val_loss':     val_m['loss'],
        'val_macro_f1': val_m['macro_f1'],
        'val_accuracy': val_m['accuracy'],
        'val_f1_real':  val_m['f1_real'],
        'val_f1_fake':  val_m['f1_fake'],
        'val_f1_nei':   val_m['f1_nei'],
    }
    history.append(row)

    f1 = val_m['macro_f1']
    improved = f1 > best_f1
    if improved:
        best_f1    = f1
        no_improve = 0
        torch.save({
            'epoch':      epoch,
            'state_dict': model.state_dict(),
            'best_f1':    best_f1,
            'config':     CONFIG,
        }, ckpt_path)
    else:
        no_improve += 1

    print(
        f'Ep {epoch:3d} | train={avg_train:.4f} | val_loss={val_m["loss"]:.4f} '
        f'| macro_F1={f1:.4f} acc={val_m["accuracy"]:.4f} '
        f'| F1 R={val_m["f1_real"]:.4f} F={val_m["f1_fake"]:.4f} N={val_m["f1_nei"]:.4f}'
        f'{" *" if improved else ""}'
    )

    if no_improve >= patience and not smoke:
        print(f'Early stopping (patience={patience})')
        break

print(f'\nBest val macro-F1: {best_f1:.4f}  ckpt: {ckpt_path}')

In [ ]:
# ── Step 11: Test Set Evaluation (load best checkpoint) ───────────────────────
best_ckpt = torch.load(ckpt_path, map_location=device)
model.load_state_dict(best_ckpt['state_dict'])
print(f'Loaded best checkpoint (epoch {best_ckpt["epoch"]}, val_f1={best_ckpt["best_f1"]:.4f})')

test_m = evaluate(model, loaders['test'], loss_cfg)

print('\n' + '=' * 60)
print('TEST RESULTS — Stage 07-C Full COOLANT (Vietnamese)')
print('=' * 60)
print(f'Test Macro-F1 : {test_m["macro_f1"]:.4f}')
print(f'Test Accuracy : {test_m["accuracy"]:.4f}')
print(f'F1-Real       : {test_m["f1_real"]:.4f}')
print(f'F1-Fake       : {test_m["f1_fake"]:.4f}')
print(f'F1-NEI        : {test_m["f1_nei"]:.4f}')
print('=' * 60)
print('\nClassification Report:')
print(classification_report(
    test_m['labels'], test_m['preds'],
    target_names=['Real', 'Fake', 'NEI'], digits=4
))

In [ ]:
# ── Step 12: Training Curves + Confusion Matrix ───────────────────────────────
df = pd.DataFrame(history)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss curve
axes[0].plot(df['epoch'], df['train_loss'], label='Train Loss', color='#3B82F6')
axes[0].plot(df['epoch'], df['val_loss'],   label='Val Loss',   color='#3B82F6', linestyle='--')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Loss'); axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

# Macro-F1 curve
axes[1].plot(df['epoch'], df['val_macro_f1'], color='#10B981', linewidth=2, label='Val Macro-F1')
axes[1].plot(df['epoch'], df['val_f1_real'],  color='#22C55E', linestyle='--', label='F1-Real')
axes[1].plot(df['epoch'], df['val_f1_fake'],  color='#EF4444', linestyle='--', label='F1-Fake')
axes[1].plot(df['epoch'], df['val_f1_nei'],   color='#A78BFA', linestyle='--', label='F1-NEI')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('F1')
axes[1].set_title('Validation F1'); axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

# Confusion matrix
cm_arr = np.array(test_m['confusion_matrix'])
sns.heatmap(
    cm_arr, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Real', 'Fake', 'NEI'],
    yticklabels=['Real', 'Fake', 'NEI'],
    ax=axes[2], cbar=False
)
axes[2].set_title(f'Test Confusion Matrix\nMacro-F1={test_m["macro_f1"]:.4f}')
axes[2].set_xlabel('Predicted'); axes[2].set_ylabel('Actual')

plt.suptitle('Stage 07-C: Full COOLANT Vietnamese Fine-Tuning', fontsize=13)
plt.tight_layout()
p = CONFIG['paths']['results_dir'] / 'training_curves.png'
plt.savefig(p, dpi=150, bbox_inches='tight'); plt.show(); plt.close()
print(f'Saved: {p}')

In [ ]:
# ── Step 13: Save Results JSON ────────────────────────────────────────────────
results = {
    'generated':       datetime.now().isoformat(),
    'best_epoch':      int(best_ckpt['epoch']),
    'best_val_macro_f1': float(best_f1),
    'test_metrics': {
        'test_macro_f1':    test_m['macro_f1'],
        'test_accuracy':    test_m['accuracy'],
        'test_f1_real':     test_m['f1_real'],
        'test_f1_fake':     test_m['f1_fake'],
        'test_f1_nei':      test_m['f1_nei'],
        'confusion_matrix': test_m['confusion_matrix'],
    },
    'training_history': history,
    'config': {
        'phobert_lr':       CONFIG['training']['phobert_lr'],
        'head_lr':          CONFIG['training']['head_lr'],
        'lambda_itm':       CONFIG['loss']['lambda_itm'],
        'lambda_cl':        CONFIG['loss']['lambda_cl'],
        'gamma_ca':         CONFIG['loss']['gamma_ca'],
        'label_smoothing':  CONFIG['loss']['label_smoothing'],
        'batch_size':       CONFIG['training']['batch_size'],
        'max_seq_len':      CONFIG['phobert']['max_seq_len'],
    },
}

rp = CONFIG['paths']['results_dir'] / 'finetune_results.json'
with open(rp, 'w') as f: json.dump(results, f, indent=2)
print(f'Results saved: {rp}')
print(f'Load in 07d for ablation comparison: {rp}')